In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load.

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md


import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q transformers datasets sentence-transformers accelerate

In [2]:
import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline
)

from sentence_transformers import SentenceTransformer

print("Torch Version:", torch.__version__)

Torch Version: 2.10.0+cpu


In [3]:
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
def combine(df):
    return (
        "Question: " + df["prompt"].fillna("") +
        " A: " + df["A"].fillna("") +
        " B: " + df["B"].fillna("") +
        " C: " + df["C"].fillna("") +
        " D: " + df["D"].fillna("") +
        " E: " + df["E"].fillna("")
    )

train_text = combine(train)
test_text = combine(test)

train_text.iloc[0]

"Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement. C: Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present. D: Martin Heidegger believes that the relations

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
sample = train_text.iloc[0]

tokens = tokenizer(
    sample,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

tokens

{'input_ids': tensor([[  101,  3160,  1024,  4060,  1996,  2190,  2825,  3437,  1024,  2054,
          2003,  3235,  2002,  5178, 13327,  1005,  1055,  3193,  2006,  1996,
          3276,  2090,  2051,  1998,  2529,  4598,  1029,  2426,  1996,  3205,
          7047,  1012,  1037,  1024,  3235,  2002,  5178, 13327,  7164,  2008,
          4286,  4839,  2306,  1037,  2051, 22961,  2008,  2003, 10709,  1998,
          2515,  2025,  2031,  1037,  4225,  2927,  2030,  2203,  1012,  1996,
          3276,  2000,  1996,  2627,  7336, 21894,  2009,  2004,  1037,  3439,
          3690,  1010,  1998,  1996,  3276,  2000,  1996,  2925,  7336,  4526,
          1037,  2088,  2008,  2097, 18094,  3458,  2028,  1005,  1055,  2219,
          2051,  1012,  1038,  1024,  3235,  2002,  5178, 13327,  7164,  2008,
          4286,  2079,  2025,  4839,  2503,  2051,  1010,  2021,  2008,  2027,
          2024,  2051,  1012,  1996,  3276,  2000,  1996,  2627,  2003,  1037,
          2556,  7073,  1997,  2383,  

In [7]:
model = AutoModel.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
with torch.no_grad():
    outputs = model(**tokens)

outputs.last_hidden_state.shape

torch.Size([1, 128, 768])

In [9]:
cls_embedding = outputs.last_hidden_state[:,0,:]

cls_embedding.shape

torch.Size([1, 768])

In [10]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
embeddings = embedding_model.encode(
    train_text[:5].tolist(),
    show_progress_bar=True
)

embeddings.shape

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(5, 384)

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    embeddings[0].reshape(1,-1),
    embeddings[1].reshape(1,-1)
)

print(similarity)

[[0.02978373]]


In [13]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
candidate_labels = [
    "Science",
    "Mathematics",
    "History",
    "Geography",
    "Computer Science"
]

result = classifier(
    train.loc[0,"prompt"],
    candidate_labels
)

result

{'sequence': "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.",
 'labels': ['History',
  'Science',
  'Computer Science',
  'Mathematics',
  'Geography'],
 'scores': [0.3245202600955963,
  0.2581675946712494,
  0.14894472062587738,
  0.134880930185318,
  0.13348647952079773]}

In [15]:
for label, score in zip(result["labels"], result["scores"]):
    print(f"{label:<20} {score:.4f}")

History              0.3245
Science              0.2582
Computer Science     0.1489
Mathematics          0.1349
Geography            0.1335
